# Experiment: Origen y adquisición del corpus Doppler

**Pregunta.** ¿De qué fuentes sale el audio crudo, con qué licencias y cómo se alinea cada corpus con las dos tareas (detección binaria vs tipo de sirena)?

**Criterio de éxito.** Inventario reproducible (local y Kaggle), tabla tarea↔dataset, y CSV de archivos detectados en `reports/tables/`.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

SEED = 7

# Local: data/raw/<corpus>
# Kaggle (Add Input clásico): /kaggle/input/<slug>
# Kaggle (datasets/user): /kaggle/input/datasets/<user>/<slug>/<slug>
IS_KAGGLE = Path("/kaggle/input").exists()

if IS_KAGGLE:
    DATA_ROOT = Path("/kaggle/input")
    FIGURES_DIR = Path("/kaggle/working/reports/figures")
    TABLES_DIR = Path("/kaggle/working/reports/tables")
else:
    here = Path.cwd().resolve()
    REPO_ROOT = here if (here / "data" / "raw").exists() else here.parent
    DATA_ROOT = REPO_ROOT / "data" / "raw"
    FIGURES_DIR = REPO_ROOT / "reports" / "figures"
    TABLES_DIR = REPO_ROOT / "reports" / "tables"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

CORPUS_SLUGS = {
    "sirennet": ("sirennet",),
    "lssiren": ("lssiren",),
    "urbansound8k": ("urbansound8k",),
}


def is_kaggle() -> bool:
    return IS_KAGGLE


def figures_dir() -> Path:
    return FIGURES_DIR


def tables_dir() -> Path:
    return TABLES_DIR


def resolve_corpus(name: str) -> Path | None:
    candidates: list[Path] = []
    for slug in CORPUS_SLUGS[name]:
        candidates.append(DATA_ROOT / slug)
        datasets = DATA_ROOT / "datasets"
        if datasets.exists():
            for user_dir in datasets.iterdir():
                if not user_dir.is_dir():
                    continue
                candidates.append(user_dir / slug)
                candidates.append(user_dir / slug / slug)
    existing = [path for path in candidates if path.exists()]
    if not existing:
        return None
    return max(existing, key=lambda path: len(path.parts))


@dataclass(frozen=True)
class CorpusPaths:
    sirennet: Path | None
    lssiren: Path | None
    urbansound8k: Path | None

    def available(self) -> dict[str, Path]:
        found = {
            "sirennet": self.sirennet,
            "lssiren": self.lssiren,
            "urbansound8k": self.urbansound8k,
        }
        return {key: path for key, path in found.items() if path is not None}


def corpus_paths() -> CorpusPaths:
    return CorpusPaths(
        sirennet=resolve_corpus("sirennet"),
        lssiren=resolve_corpus("lssiren"),
        urbansound8k=resolve_corpus("urbansound8k"),
    )


print("kaggle:", IS_KAGGLE)
print("DATA_ROOT:", DATA_ROOT)
print("FIGURES_DIR:", FIGURES_DIR)
print("TABLES_DIR:", TABLES_DIR)
print("available:", list(corpus_paths().available()))
SEED


In [2]:
# Inventario de archivos
"""Inventario de archivos de audio del corpus Doppler."""

from __future__ import annotations

from pathlib import Path

import pandas as pd

SIRENNET_CLASS_MAP = {
    "ambulance": "ambulance",
    "police": "police",
    "firetruck": "firetruck",
    "fire_truck": "firetruck",
    "fire": "firetruck",
    "traffic": "traffic",
}

LSSIREN_POSITIVE_HINTS = ("emergency", "siren", "ambulance")
LSSIREN_NEGATIVE_HINTS = ("road", "noise", "traffic")
AUDIO_SUFFIXES = {".wav", ".mp3", ".flac", ".ogg", ".m4a"}


def is_audio(path: Path) -> bool:
    return path.suffix.lower() in AUDIO_SUFFIXES


def infer_sirennet_label(path: Path) -> str | None:
    parts = [p.lower() for p in path.parts]
    stem = path.stem.lower()
    for key, label in SIRENNET_CLASS_MAP.items():
        if key in parts or stem.startswith(key) or f"_{key}_" in f"_{stem}_":
            return label
    return None


def infer_lssiren_label(path: Path) -> str | None:
    blob = " ".join(p.lower() for p in path.parts)
    if any(h in blob for h in LSSIREN_POSITIVE_HINTS) and "road" not in blob:
        return "siren"
    if any(h in blob for h in LSSIREN_NEGATIVE_HINTS):
        return "road_noise"
    return None


def scan_sirennet(root: Path) -> pd.DataFrame:
    rows = []
    for path in root.rglob("*"):
        if not path.is_file() or not is_audio(path):
            continue
        rows.append(
            {
                "corpus": "sirennet",
                "path": str(path),
                "relpath": str(path.relative_to(root)),
                "label": infer_sirennet_label(path) or "unknown",
                "task": "multiclass",
            }
        )
    return pd.DataFrame(rows)


LSSIREN_CSV_COLUMNS = [
    "filename",
    "chroma_stft",
    "rmse",
    "spectral_centroid",
    "spectral_bandwidth",
    "rolloff",
    "zero_crossing_rate",
    *[f"mfcc{i}" for i in range(1, 21)],
    "label",
]


def read_lssiren_features(path: Path) -> pd.DataFrame:
    raw = pd.read_csv(path)
    if "filename" in raw.columns:
        return raw
    return pd.read_csv(path, header=None, names=LSSIREN_CSV_COLUMNS)


def scan_lssiren(root: Path) -> pd.DataFrame:
    rows = []
    for path in root.rglob("*"):
        if not path.is_file() or not is_audio(path):
            continue
        label = infer_lssiren_label(path) or "unknown"
        rows.append(
            {
                "corpus": "lssiren",
                "path": str(path),
                "relpath": str(path.relative_to(root)),
                "label": label,
                "task": "binary",
            }
        )
    if rows:
        return pd.DataFrame(rows)

    for csv_path in root.glob("*.csv"):
        feat = read_lssiren_features(csv_path)
        label_col = "label" if "label" in feat.columns else feat.columns[-1]
        name_col = "filename" if "filename" in feat.columns else feat.columns[0]
        for _, row in feat.iterrows():
            raw_label = str(row[label_col]).strip().lower()
            label = "siren" if raw_label in {"ambulance", "siren", "emergency"} else "road_noise"
            rows.append(
                {
                    "corpus": "lssiren",
                    "path": str(csv_path.parent / str(row[name_col])),
                    "relpath": str(row[name_col]),
                    "label": label,
                    "task": "binary",
                    "source": "feature_csv",
                }
            )
    return pd.DataFrame(rows)


def scan_urbansound8k(root: Path) -> pd.DataFrame:
    csv_candidates = list(root.rglob("UrbanSound8K.csv"))
    if csv_candidates:
        meta = pd.read_csv(csv_candidates[0])
        if "class" not in meta.columns and "class_name" in meta.columns:
            meta = meta.rename(columns={"class_name": "class"})
        audio_root = csv_candidates[0].parent.parent / "audio"
        if not audio_root.exists():
            audio_root = root / "audio"
            if not audio_root.exists():
                audio_root = root
        rows = []
        for _, row in meta.iterrows():
            fold = int(row["fold"])
            fname = row["slice_file_name"]
            path = audio_root / f"fold{fold}" / fname
            rows.append(
                {
                    "corpus": "urbansound8k",
                    "path": str(path),
                    "relpath": f"fold{fold}/{fname}",
                    "label": row["class"],
                    "task": "urban_scene",
                    "fold": fold,
                    "fsID": row.get("fsID"),
                    "classID": row.get("classID"),
                    "salience": row.get("salience"),
                }
            )
        return pd.DataFrame(rows)

    rows = []
    for path in root.rglob("*"):
        if not path.is_file() or not is_audio(path):
            continue
        rows.append(
            {
                "corpus": "urbansound8k",
                "path": str(path),
                "relpath": str(path.relative_to(root)),
                "label": path.parent.name,
                "task": "urban_scene",
            }
        )
    return pd.DataFrame(rows)


## Plan

- Hipótesis: ningún dataset único cubre tipo (4 clases) + in-the-wild + distractores urbanos.
- Barrido: sireNNet, LSSiren, UrbanSound8K (trabajo) y AudioSet-EV v2 (escala, no descargado aquí).
- Métricas de esta notebook: n_archivos, clases, presencia/ausencia de WAV, licencia.


In [4]:
import pandas as pd

FIG = figures_dir()
TAB = tables_dir()

catalog = pd.DataFrame([
    {
        "corpus": "sireNNet",
        "role": "primario_4clases",
        "task": "binary+multiclass",
        "n_nominal": 1675,
        "classes": "ambulance, police, firetruck, traffic",
        "license": "CC BY 4.0",
        "source": "https://data.mendeley.com/datasets/j4ydzzv4kb/1",
        "citation": "Shah & Singh, 2023, 10.17632/j4ydzzv4kb.1",
        "notes": "Release publico ya aumentado; no hay split original vs augment",
    },
    {
        "corpus": "LSSiren",
        "role": "binario_in_the_wild",
        "task": "binary",
        "n_nominal": 1800,
        "classes": "siren, road_noise",
        "license": "CC BY 4.0",
        "source": "https://doi.org/10.6084/m9.figshare.19291472",
        "citation": "Asif et al., Sci Data 2022, 10.1038/s41597-022-01727-2",
        "notes": "Karachi + setup experimental + internet; sesgo a ambulancia",
    },
    {
        "corpus": "UrbanSound8K",
        "role": "distractores_urbanos",
        "task": "urban_scene / binary_generic_siren",
        "n_nominal": 8732,
        "classes": "10 clases urbanas incl. siren (~929)",
        "license": "CC BY-NC 4.0",
        "source": "https://zenodo.org/records/1203745",
        "citation": "Salamon, Jacoby & Bello, 2014, 10.5281/zenodo.1203745",
        "notes": "Usar folds oficiales; no partir el mismo fsID",
    },
    {
        "corpus": "AudioSet-EV v2",
        "role": "escala_entrenamiento_posterior",
        "task": "binary+multiclass",
        "n_nominal": 28816,
        "classes": "police / ambulance / fire vs urban negatives",
        "license": "subset AudioSet (YouTube); uso de investigacion",
        "source": "https://doi.org/10.5281/zenodo.18668076",
        "citation": "Giacomelli & Rinaldi, 2025, 10.5281/zenodo.18668076",
        "notes": "~8-16 GB zip / ~28 GB WAV; no se descarga en esta fase",
    },
])
catalog.to_csv(TAB / "corpus_catalog.csv", index=False)
catalog


,corpus,role,task,n_nominal,classes,license,source,citation,notes
0,sireNNet,primario_4clases,binary+multiclass,1675,"ambulance, police, firetruck, traffic",CC BY 4.0,https://data.mendeley.com/datasets/j4ydzzv4kb/1,"Shah & Singh, 2023, 10.17632/j4ydzzv4kb.1",Release publico ya aumentado; no hay split ori...
1,LSSiren,binario_in_the_wild,binary,1800,"siren, road_noise",CC BY 4.0,https://doi.org/10.6084/m9.figshare.19291472,"Asif et al., Sci Data 2022, 10.1038/s41597-022...",Karachi + setup experimental + internet; sesgo...
2,UrbanSound8K,distractores_urbanos,urban_scene / binary_generic_siren,8732,10 clases urbanas incl. siren (~929),CC BY-NC 4.0,https://zenodo.org/records/1203745,"Salamon, Jacoby & Bello, 2014, 10.5281/zenodo....",Usar folds oficiales; no partir el mismo fsID
3,AudioSet-EV v2,escala_entrenamiento_posterior,binary+multiclass,28816,police / ambulance / fire vs urban negatives,subset AudioSet (YouTube); uso de investigacion,https://doi.org/10.5281/zenodo.18668076,"Giacomelli & Rinaldi, 2025, 10.5281/zenodo.186...",~8-16 GB zip / ~28 GB WAV; no se descarga en e...


## Alineación tarea ↔ corpus

El protocolo de splits evita mezclar orígenes en el mismo fold de entrenamiento sin declararlo.


In [5]:
alignment = pd.DataFrame([
    {"task": "deteccion_binaria", "train": "sireNNet (sirena vs traffic)", "test_internal": "holdout sireNNet", "test_external": "LSSiren; UrbanSound8K siren vs resto"},
    {"task": "tipo_multiclase", "train": "sireNNet (ambulance/police/firetruck)", "test_internal": "holdout sireNNet agrupado", "test_external": "AudioSet-EV eval (fase posterior)"},
    {"task": "distractores_duros", "train": "UrbanSound8K no-siren (folds 1-8)", "test_internal": "folds 9-10", "test_external": "LSSiren road_noise"},
])
alignment.to_csv(TAB / "task_alignment.csv", index=False)
alignment


,task,train,test_internal,test_external
0,deteccion_binaria,sireNNet (sirena vs traffic),holdout sireNNet,LSSiren; UrbanSound8K siren vs resto
1,tipo_multiclase,sireNNet (ambulance/police/firetruck),holdout sireNNet agrupado,AudioSet-EV eval (fase posterior)
2,distractores_duros,UrbanSound8K no-siren (folds 1-8),folds 9-10,LSSiren road_noise


## Inventario de archivos montados

Si un corpus no está, la notebook no falla: registra `missing` y sigue.


In [6]:

paths = corpus_paths()
frames = []
status_rows = []

scanners = {
    "sirennet": (paths.sirennet, scan_sirennet),
    "lssiren": (paths.lssiren, scan_lssiren),
    "urbansound8k": (paths.urbansound8k, scan_urbansound8k),
}

for name, (root, scanner) in scanners.items():
    if root is None:
        status_rows.append({"corpus": name, "status": "missing", "root": None, "n_rows": 0})
        continue
    df = scanner(root)
    frames.append(df)
    status_rows.append({"corpus": name, "status": "ok", "root": str(root), "n_rows": int(len(df))})

status = pd.DataFrame(status_rows)
status.to_csv(TAB / "mount_status.csv", index=False)
inventory = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=["corpus", "path", "relpath", "label", "task"])
if not inventory.empty:
    inventory.to_csv(TAB / "file_inventory.csv", index=False)
status


,corpus,status,root,n_rows
0,sirennet,ok,/home/jeancdevx/dev/doppler/doppler-ml/data/ra...,1675
1,lssiren,ok,/home/jeancdevx/dev/doppler/doppler-ml/data/ra...,1834
2,urbansound8k,ok,/home/jeancdevx/dev/doppler/doppler-ml/data/ra...,8732


In [ ]:
if inventory.empty:
    counts = pd.DataFrame(columns=["corpus", "label", "n"])
else:
    counts = inventory.groupby(["corpus", "label"]).size().reset_index(name="n")
counts.to_csv(TAB / "label_counts.csv", index=False)
counts


/home/jeancdevx/dev/doppler/doppler-ml/data/raw /home/jeancdevx/dev/doppler/doppler-ml/data/raw/sirennet


## Resultados

- El catálogo y el protocolo de splits quedan fijados antes de entrenar.
- `file_inventory.csv` es la tabla semiestructurada que alimenta las notebooks 02 y 03.
- Siguiente: exploración acústica (duración, sample rate, formas de onda).


In [ ]:
result = {
    "seed": SEED,
    "n_catalog_rows": int(len(catalog)),
    "mounted": status.set_index("corpus")["status"].to_dict(),
    "n_inventory_rows": int(len(inventory)),
    "tables": [p.name for p in sorted(TAB.glob("*.csv"))],
}
result


{'seed': 7,
 'n_catalog_rows': 4,
 'mounted': {'sirennet': 'ok', 'lssiren': 'ok', 'urbansound8k': 'ok'},
 'n_inventory_rows': 12241,
 'tables': ['corpus_catalog.csv',
  'file_inventory.csv',
  'label_counts.csv',
  'mount_status.csv',
  'task_alignment.csv']}